In [8]:
import sys, pathlib
repo_root = pathlib.Path("/Users/muhammadnumanmuttaqi/Documents/MScITBE/Thesis/Thesis/Geometry-to-Ontology")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [12]:
from rdflib import Graph
from owlready2 import get_ontology, sync_reasoner_pellet, default_world
import pathlib, tempfile

data_core       = "../ontology/core/Resplan.ttl"
data_floorplan  = "../output/imp_resplan_ttl/plan_00000_drop_door.ttl"
swrl_rules      = "../ontology/rules/DoorRule.swrl.ttl"


In [18]:
print("=== Step 1: Merging Turtle files ===")
# Combining graphs
merged_graph = Graph()
for src in (data_core, data_floorplan, swrl_rules):
    print(f"Loading: {src}")
    merged_graph.parse(src, format="turtle")

print(f"Total triples loaded: {len(merged_graph)}")

# Save as RDF/XML instead of Turtle
tmp = pathlib.Path(tempfile.gettempdir()) / "merged_for_reasoning.owl"
print(f"\n=== Step 2: Saving merged graph as RDF/XML ===")
print(f"Temp file: {tmp}")
merged_graph.serialize(tmp, format="xml")  # ← CHANGED from "turtle" to "xml"

# Verify the file is valid
print("\n=== Step 3: Validating RDF/XML file ===")
test_graph = Graph()
test_graph.parse(tmp, format="xml")
print(f"Validation OK: {len(test_graph)} triples")

# Load with proper format specification
print("\n=== Step 4: Loading into Owlready2 ===")
onto = get_ontology(tmp.as_uri()).load()  # Owlready2 auto-detects .owl as RDF/XML

print("\n=== Step 5: Running Pellet reasoner with SWRL ===")
with onto:
    sync_reasoner_pellet(
        infer_property_values=True, 
        infer_data_property_values=True,
        debug=1  # Add debug output to see what's happening
    )

print("\n=== Step 6: Saving inferred ontology ===")
# Save output
inferred_path = "../output/inferred_resplan_ttl/plan_00000_inferred_drop_door.owl"
onto.save(file=inferred_path, format="rdfxml")
print(f"Saved (RDF/XML): {inferred_path}")

=== Step 1: Merging Turtle files ===
Loading: ../ontology/core/Resplan.ttl
Loading: ../output/imp_resplan_ttl/plan_00000_drop_door.ttl
Loading: ../ontology/rules/DoorRule.swrl.ttl
Total triples loaded: 641

=== Step 2: Saving merged graph as RDF/XML ===
Temp file: /var/folders/_r/pq1zmsn93blcqytv44mnlsrm0000gn/T/merged_for_reasoning.owl

=== Step 3: Validating RDF/XML file ===
Validation OK: 641 triples

=== Step 4: Loading into Owlready2 ===

=== Step 5: Running Pellet reasoner with SWRL ===


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp /Users/muhammadnumanmuttaqi/anaconda3/envs/cubi-floortrans/lib/python3.11/site-packages/owlready2/pellet/httpclient-4.2.3.jar:/Users/muhammadnumanmuttaqi/anaconda3/envs/cubi-floortrans/lib/python3.11/site-packages/owlready2/pellet/aterm-java-1.6.jar:/Users/muhammadnumanmuttaqi/anaconda3/envs/cubi-floortrans/lib/python3.11/site-packages/owlready2/pellet/xercesImpl-2.10.0.jar:/Users/muhammadnumanmuttaqi/anaconda3/envs/cubi-floortrans/lib/python3.11/site-packages/owlready2/pellet/slf4j-api-1.6.4.jar:/Users/muhammadnumanmuttaqi/anaconda3/envs/cubi-floortrans/lib/python3.11/site-packages/owlready2/pellet/jena-tdb-0.10.0.jar:/Users/muhammadnumanmuttaqi/anaconda3/envs/cubi-floortrans/lib/python3.11/site-packages/owlready2/pellet/jena-iri-0.9.5.jar:/Users/muhammadnumanmuttaqi/anaconda3/envs/cubi-floortrans/lib/python3.11/site-packages/owlready2/pellet/owlapi-distribution-3.4.3-bin.jar:/Users/muhammadnumanmuttaqi/anaconda3/envs/cubi-floortr


=== Step 6: Saving inferred ontology ===
Saved (RDF/XML): ../output/inferred_resplan_ttl/plan_00000_inferred_drop_door.owl


* Owlready2 * Pellet took 1.6754491329193115 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [19]:
# Convert back to Turtle for readability
print("\n=== Step 7: Converting output to Turtle ===")
final_graph = Graph()
final_graph.parse(inferred_path, format="xml")
turtle_output = inferred_path.replace(".owl", ".ttl")
final_graph.serialize(turtle_output, format="turtle")
print(f"Saved (Turtle): {turtle_output}")

print("\n=== Inference Complete ===")


=== Step 7: Converting output to Turtle ===
Saved (Turtle): ../output/inferred_resplan_ttl/plan_00000_inferred_drop_door.ttl

=== Inference Complete ===
